# 🪳 Exercise - Debugging a RAG Failure

## Goal

Learn to diagnose **where a bad RAG answer failed** instead of immediately changing the LLM or rewriting the prompt.

## Scenario

You are debugging a support assistant that answers questions using internal policy documents. For each case below, identify the **earliest important failure in the RAG pipeline**.

Use these categories:

| Failure category | Meaning |
|---|---|
| **ingestion problem** | the required source was missing, stale, or badly parsed |
| **chunking problem** | relevant information was split so the retrieved chunk lost necessary meaning |
| **retrieval problem** | the retriever selected the wrong chunks |
| **filtering problem** | metadata/access filters excluded useful evidence |
| **context problem** | the right evidence existed, but the model received too much, too little, or conflicting context |
| **prompt problem** | instructions did not clearly require grounded behavior |
| **generation problem** | sufficient evidence and instructions were present, but the model still produced an unsupported answer |
| **no-answer case** | the knowledge base does not contain enough evidence to answer safely |

## Cases

### Case A - The missing policy page

**User question:** "How many days do I have to request a refund for the Pro plan?""

**Retrieved chunks:**
- "Basic plan cancellations can be requested within 14 days."
- "Refund requests are submitted from the Billing page."

**Final answer:** "You have 14 days to request a refund."

**Expected behavior:** The system should not apply the Basic-plan rule to the Pro plan. The current Pro-plan refund policy exists in the source repository, but its page produced no text during PDF extraction.

---

### Case B - The condition was split away

**User question:** "Do I pay a cancellation fee if I cancel during the trial?"

**Retrieved chunk:** "A €25 cancellation fee applies to cancellations before the end of the annual commitment."

A neighboring chunk, which was **not retrieved**, says: "The fee does not apply while the account is still in the 30-day trial period."

**Final answer:** "Yes. The cancellation fee is €25."

**Expected behavior:** The answer should mention the trial exception.

---

### Case C - Semantically similar, operationally wrong

**User question:** "How do I change the email address that receives invoices?"

**Retrieved chunks:**
- "Marketing email preferences can be changed under Notifications."
- "You can unsubscribe from promotional email from any marketing message."

**Final answer:** "Open Notifications and change your email preferences."

**Expected behavior:** The assistant should retrieve the billing-profile instructions, not marketing-preference documentation.

---

### Case D - A filter removed the right policy

**User question:** "Can an EU customer receive a cash refund?"

**Retriever configuration:** `region="US"`

**Retrieved chunk:** "US customers receive refunds to the original payment method."

**Final answer:** "Yes. The refund is returned to the original payment method."

**Expected behavior:** The customer is in the EU, so the EU policy should be retrieved. The current metadata filter prevents that source from being considered.

---

### Case E - Good evidence, unsupported generation

**User question:** "Can you issue the refund automatically?"

**Retrieved chunk:** "Refunds above €100 require approval by a billing specialist before any refund action is executed."

**Prompt instruction:** "Answer the user helpfully using the context below."

**Final answer:** "Yes. I can issue the refund now."

**Expected behavior:** The assistant should state that approval is required and should not claim that the action has been completed.

---

### Case F - The corpus does not answer it

**User question:** "What warranty do I get when I buy your product from an unaffiliated third-party marketplace?"

**Retrieved chunks:**
- direct-purchase return policy,
- shipping policy,
- subscription cancellation policy.

**Final answer:** "Third-party purchases include a one-year warranty."

**Expected behavior:** The knowledge base contains no third-party warranty policy. The system should say that it cannot verify the answer from its sources and route the user to an appropriate fallback.

## Task

For each case:

1. identify the primary failure category,
2. identify the evidence that led you to that diagnosis,
3. propose **one targeted fix**,
4. explain one tradeoff or side effect of the fix,
5. decide whether the current system should **answer, refuse, ask for clarification, or escalate**.

## Deliverables

Complete a table like this:

| Case | Failure category | Evidence | Targeted fix | Tradeoff | Safe behavior now |
|---|---|---|---|---|---|
| A |  |  |  |  |  |
| B |  |  |  |  |  |
| C |  |  |  |  |  |
| D |  |  |  |  |  |
| E |  |  |  |  |  |
| F |  |  |  |  |  |

## Hints

- Start at the **left side of the RAG pipeline**. If the required source never entered the usable corpus, changing the prompt cannot fix the root cause.
- "The answer is wrong" is a symptom, not a diagnosis.
- Prefer the **smallest fix that targets the failing stage**.
- A good RAG system must have a useful behavior for "the answer is not in the corpus."

## Solution

Suggested diagnosis:

| Case | Failure category | Evidence | Targeted fix | Tradeoff | Safe behavior now |
|---|---|---|---|---|---|
| **A** | ingestion problem | the Pro-plan source exists, but extraction produced no usable text | fix/replace the extractor and add an ingestion check for empty or suspiciously short documents | better extraction may cost more time and require format-specific tooling | refuse to infer the Pro rule; escalate or state that the policy cannot be verified |
| **B** | chunking problem | the fee and its trial exception were separated, and only one half was retrieved | increase contextual overlap or chunk by policy section rather than fixed size | larger/overlapping chunks increase index size and can add irrelevant context | do not give an unconditional "yes"; retrieve more context or escalate |
| **C** | retrieval problem | retrieved chunks are about marketing email rather than invoice delivery | improve retrieval with better query formulation, hybrid search, reranking, or stronger document metadata | extra retrieval stages increase latency and complexity | avoid answering from irrelevant evidence; retry retrieval |
| **D** | filtering problem | the retriever was explicitly limited to `region="US"` for an EU customer | derive the region filter from trusted customer/account context and test filter logic | stricter metadata dependencies can reduce recall when metadata is missing | ask for/verify region if needed, then rerun retrieval |
| **E** | prompt problem / generation problem | the correct policy is present, but the prompt does not strongly require grounded, action-aware behavior; the generated answer contradicts the evidence | require source-grounded answers and explicit handling of actions/approval; validate claims before executing tools | stricter prompts may produce more cautious answers and more escalations | state that billing approval is required; do not claim the refund was issued |
| **F** | no-answer case | none of the retrieved or available sources defines third-party warranty coverage | add an explicit no-answer policy and route unsupported questions to a fallback; add the policy source only if the organization actually has one | more refusals/escalations when coverage is incomplete | say the answer cannot be verified from available policy and escalate/redirect |

The important habit is to **diagnose before tuning**. A prompt change cannot repair missing documents, and a new embedding model cannot fix an incorrect metadata filter.

</details>


<hr/>